# Amazon Sales EDA and SQL Analysis

This notebook mirrors the Streamlit dashboard workflow: load the Amazon sales report, clean it, and run SQL-backed analytics.

In [ ]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

project_root = Path('..').resolve()
csv_path = project_root / 'Amazon Sale Report.csv'
if not csv_path.exists():
    csv_path = project_root / 'dataset' / 'Amazon Sale Report.csv'

df = pd.read_csv(csv_path, low_memory=False)
df.head()

In [ ]:
def clean_sales_data(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.dropna(axis=1, how='all').copy()
    frame.columns = [str(column).strip().lower().replace(' ', '_') for column in frame.columns]
    frame['date'] = pd.to_datetime(frame['date'], errors='coerce')
    frame['amount'] = pd.to_numeric(frame['amount'], errors='coerce').fillna(0)
    frame['qty'] = pd.to_numeric(frame['qty'], errors='coerce').fillna(0).astype(int)
    frame['b2b'] = frame['b2b'].astype(str).str.lower().isin(['true', '1', 'yes'])
    return frame.dropna(subset=['date'])

clean_df = clean_sales_data(df)
connection = sqlite3.connect(':memory:')
clean_df.to_sql('amazon_sales', connection, index=False, if_exists='replace')

sql = '''
SELECT strftime('%Y-%m', date) AS month, SUM(amount) AS revenue
FROM amazon_sales
GROUP BY month
ORDER BY month
'''
monthly_sales = pd.read_sql_query(sql, connection)
monthly_sales

In [ ]:
fig = px.line(monthly_sales, x='month', y='revenue', markers=True, title='Monthly Revenue Trend')
fig.show()